# Logistic Regression — Network Intrusion Detection

Huấn luyện và đánh giá mô hình **Logistic Regression** trên tập dữ liệu CICIDS2017.
Logistic Regression yêu cầu feature scaling nên ta dùng `StandardScaler` trước khi train.

In [ ]:
import os
import sys
import time
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Thêm thư mục gốc dự án vào sys.path
sys.path.insert(0, str(Path.cwd()))
sys.path.append(os.path.abspath(".."))

from src.model_training import (
    load_splits,
    evaluate_model,
    plot_confusion_matrix,
    compare_models,
)

In [ ]:
# --- STEP 1: Load dữ liệu đã chia sẵn ---
print("=" * 70)
print("STEP 1: Loading pre-split data")
print("=" * 70)

X_train, X_test, y_train, y_test = load_splits()

print(f"Feature columns: {X_train.shape[1]}")
print(f"Train size: {len(X_train):,}  |  Test size: {len(X_test):,}")
print(f"Classes: {sorted(y_train.unique())}")

In [ ]:
# --- STEP 2: Feature scaling ---
print("\n" + "=" * 70)
print("STEP 2: Feature Scaling (StandardScaler)")
print("=" * 70)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling applied to training and test sets.")

In [ ]:
# --- STEP 3: Train Logistic Regression ---
print("\n" + "=" * 70)
print("STEP 3: Training Logistic Regression")
print("=" * 70)

t0 = time.time()

lr_model = LogisticRegression(
    max_iter=1000,
    solver="lbfgs",
    random_state=42,
    n_jobs=-1,
)

print("Fitting model on scaled training data...")
lr_model.fit(X_train_scaled, y_train)

train_time = time.time() - t0
print(f"\nTraining completed in {train_time:.2f} seconds")
print(f"Iterations per class: {lr_model.n_iter_}")

In [ ]:
# --- STEP 4: Evaluate ---
print("\n" + "=" * 70)
print("STEP 4: Model Evaluation")
print("=" * 70)

y_pred = lr_model.predict(X_test_scaled)
y_pred_proba = lr_model.predict_proba(X_test_scaled)

lr_results = evaluate_model(
    y_true=y_test,
    y_pred=y_pred,
    model_name="Logistic Regression",
    y_pred_proba=y_pred_proba,
    labels=lr_model.classes_.tolist(),
)

In [ ]:
# --- STEP 5: Confusion Matrix ---
print("\n" + "=" * 70)
print("STEP 5: Confusion Matrix")
print("=" * 70)

plot_confusion_matrix(
    y_true=y_test,
    y_pred=y_pred,
    labels=lr_model.classes_.tolist(),
    model_name="Logistic Regression",
    normalize=True,
    figsize=(12, 10),
)

In [ ]:
# --- STEP 6: Summary ---
print("\n" + "=" * 70)
print("STEP 6: Summary")
print("=" * 70)

comparison_df = compare_models([lr_results])
print("\nPerformance Metrics:")
print(comparison_df.to_string(index=False))

print(f"\nTraining time : {train_time:.2f}s")
print(f"Test set size : {len(y_test):,}")
print(f"Features      : {X_train.shape[1]}")
print(f"Classes       : {len(lr_model.classes_)}")